In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

In [2]:
df = pd.read_csv("../../data/emi_prediction_dataset_cleaned.csv")
df.shape

(398455, 31)

In [3]:
X = df.drop(['emi_eligibility', 'max_monthly_emi'], axis=1)
y = df['emi_eligibility']

In [4]:
cat_cols = X.select_dtypes(include=['object']).columns
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X.head()

,age,monthly_salary,years_of_employment,monthly_rent,family_size,dependents,school_fees,college_fees,travel_expenses,groceries_utilities,...,company_type_Mid-size,company_type_Small,company_type_Startup,house_type_Own,house_type_Rented,existing_loans_Yes,emi_scenario_Education EMI,emi_scenario_Home Appliances EMI,emi_scenario_Personal Loan EMI,emi_scenario_Vehicle EMI
0,38.0,82600.0,0.9,20000.0,3,2,0.0,0.0,7200.0,19500.0,...,True,False,False,False,True,True,False,False,True,False
1,38.0,21500.0,7.0,0.0,2,1,5100.0,0.0,1400.0,5400.0,...,False,False,False,False,False,True,False,False,False,False
2,38.0,86100.0,5.8,0.0,4,3,0.0,0.0,10200.0,19400.0,...,False,False,True,True,False,False,True,False,False,False
3,58.0,66800.0,2.2,0.0,5,4,11400.0,0.0,6200.0,11900.0,...,True,False,False,True,False,False,False,False,False,True
4,48.0,57300.0,3.4,0.0,4,3,9400.0,21300.0,3600.0,16200.0,...,True,False,False,False,False,False,False,True,False,False


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
param_grid = {
    "n_estimators": [200, 300],
    "max_depth": [10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

rf_model = RandomForestClassifier(random_state=42, class_weight="balanced")
 
random_search = RandomizedSearchCV(
    rf_model,
    param_distributions= param_grid,
    n_iter= 20,
    cv= 3,
    scoring="f1_weighted",
    n_jobs= -1,
    verbose=2
)

random_search.fit(X_train, y_train)
best_params = random_search.best_params_

print(best_params)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [10]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)

In [12]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

roc_auc_rf = roc_auc_score(y_test, y_prob, multi_class="ovr")
print("ROC AUC Score:", roc_auc_rf)

Accuracy: 0.880111932338658

Classification Report:
               precision    recall  f1-score   support

    Eligible       0.85      0.85      0.85     14764
   High_Risk       0.23      0.52      0.32      3473
Not_Eligible       0.98      0.91      0.94     61454

    accuracy                           0.88     79691
   macro avg       0.68      0.76      0.70     79691
weighted avg       0.92      0.88      0.90     79691


Confusion Matrix:
 [[12578  1696   490]
 [ 1105  1821   547]
 [ 1164  4552 55738]]
ROC AUC Score: 0.9540517483513046
